In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
SEED = 81


def seed_everything(seed=5):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


seed_everything(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[Hardware] Active compute device: {DEVICE}")

TRAIN_DATA_PATH = "/kaggle/input/competitions/co-5420-air-pollution-forecasting-using-temporal-n-ns/CO5420-AirPollution/public/train_raw.csv"
TEST_DATA_PATH = "/kaggle/input/competitions/co-5420-air-pollution-forecasting-using-temporal-n-ns/CO5420-AirPollution/public/test.csv"
TEST_RAW_DATA_PATH = "/kaggle/input/competitions/co-5420-air-pollution-forecasting-using-temporal-n-ns/CO5420-AirPollution/public/test_raw.csv"

WD_MAPPING = {
    "N": 0,
    "NNE": 22.5,
    "NE": 45,
    "ENE": 67.5,
    "E": 90,
    "ESE": 112.5,
    "SE": 135,
    "SSE": 157.5,
    "S": 180,
    "SSW": 202.5,
    "SW": 225,
    "WSW": 247.5,
    "W": 270,
    "WNW": 292.5,
    "NW": 315,
    "NNW": 337.5,
    "CV": 0,
}

TARGET = "PM2.5"
SEQ_LENGTH = 24
N_FOLDS = 5
PURGE_GAP_HOURS = 24

# GRU CONFIG
CONTINUOUS_FEATURES_GRU = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "U_wind",
    "V_wind",
    "rel_humidity",
    "dispersion_idx",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "regional_pm25_mean",
    "station_regional_delta",
]
MISSING_TARGETS = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
]
MISSING_FEATURES_GRU = [f"{feat}_is_missing" for feat in MISSING_TARGETS]
FEATURES_GRU = CONTINUOUS_FEATURES_GRU + MISSING_FEATURES_GRU

BATCH_SIZE = 512
HIDDEN_SIZE_GRU = 256
NUM_LAYERS_GRU = 4
EPOCHS_GRU = 30
LR_GRU = 0.001
PATIENCE_GRU = 6

# LIGHTGBM CONFIG
LAG_FEATURES = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM",
    "U_wind",
    "V_wind",
    "rel_humidity",
]
CONTINUOUS_FEATURES_LGB = [
    "PM2.5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "U_wind",
    "V_wind",
    "rel_humidity",
    "dispersion_idx",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]

print("Done: Imports and Configuration loaded!")

[Hardware] Active compute device: cuda:0
Done: Imports and Configuration loaded!


In [2]:
# ==========================================
# 2. DATA PREPROCESSING
# ==========================================
print("\n" + "=" * 50)
print(" 1. ADVANCED FEATURE ENGINEERING & REGIONAL SPATIAL METRICS")
print("=" * 50)
print("Starting: Train dataset preprocessing and GRU sequence generation...")

def compute_relative_humidity(temp, dewp):
    temp_c = np.clip(temp, -40, 50)
    dewp_c = np.clip(dewp, -50, 40)
    numerator = 17.625 * dewp_c
    denominator = 243.04 + dewp_c
    sat_vap_dewp = np.exp(numerator / denominator)
    sat_vap_temp = np.exp((17.625 * temp_c) / (243.04 + temp_c))
    rh = 100.0 * (sat_vap_dewp / (sat_vap_temp + 1e-6))
    return np.clip(rh, 0.0, 100.0)


def engineer_features_narrow(df):
    df = df.copy()
    angles_rad = np.radians(df["wd"].map(WD_MAPPING).fillna(0))
    wspm = df["WSPM"].fillna(0)
    df["U_wind"] = -wspm * np.sin(angles_rad)
    df["V_wind"] = -wspm * np.cos(angles_rad)

    df["rel_humidity"] = compute_relative_humidity(
        df["TEMP"].fillna(0), df["DEWP"].fillna(0)
    )
    df["dispersion_idx"] = wspm * df["PM2.5"].fillna(0)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12.0)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12.0)

    for feat in MISSING_TARGETS:
        df[f"{feat}_is_missing"] = df[feat].isna().astype(float)

    return df


train_df = pd.read_csv(TRAIN_DATA_PATH)
train_df["datetime"] = pd.to_datetime(train_df[["year", "month", "day", "hour"]])
train_df.sort_values(by=["station", "datetime"], inplace=True)
train_df.reset_index(drop=True, inplace=True)

train_df = engineer_features_narrow(train_df)

regional_pm25 = train_df.groupby("datetime")["PM2.5"].transform("mean")
train_df["regional_pm25_mean"] = regional_pm25.ffill().bfill()
train_df["station_regional_delta"] = train_df["PM2.5"] - train_df["regional_pm25_mean"]

train_df["station"] = train_df["station"].astype("category")
station_map = {s: i for i, s in enumerate(train_df["station"].cat.categories)}
train_df["station_code"] = train_df["station"].map(station_map)

for feat in CONTINUOUS_FEATURES_LGB + ["regional_pm25_mean", "station_regional_delta"]:
    if feat in train_df.columns:
        train_df[feat] = train_df.groupby("station", observed=False)[feat].transform(
            lambda x: x.ffill().bfill()
        )

for feat in MISSING_FEATURES_GRU:
    train_df[feat] = train_df[feat].fillna(0.0)


# Create 24-hour sequence dataset for GRU
def create_sequences_gru(data, seq_length=24):
    xs, ys, times, stats = [], [], [], []
    feature_arr = data[FEATURES_GRU].values.astype(np.float32)
    target_arr = data[TARGET].values.astype(np.float32)
    time_arr = data["datetime"].values
    stat_arr = data["station_code"].values

    for i in range(len(data) - seq_length):
        xs.append(feature_arr[i : i + seq_length])
        ys.append(target_arr[i + seq_length])
        times.append(time_arr[i + seq_length])
        stats.append(stat_arr[i + seq_length])

    return np.array(xs), np.array(ys), np.array(times), np.array(stats)


print("  - Creating GRU sequences for each station...")
X_seq_list, y_seq_list, time_seq_list, stat_seq_list = [], [], [], []
for i, station in enumerate(train_df["station"].unique()):
    st_df = train_df[train_df["station"] == station]
    X_s, y_s, t_s, s_s = create_sequences_gru(st_df, SEQ_LENGTH)
    X_seq_list.append(X_s)
    y_seq_list.append(y_s)
    time_seq_list.append(t_s)
    stat_seq_list.append(s_s)
    if (i + 1) % 4 == 0:
        print(f"    ... processed {i + 1}/{len(train_df['station'].unique())} stations")

X_train_raw = np.vstack(X_seq_list)
y_train_raw = np.concatenate(y_seq_list)
times_raw = np.concatenate(time_seq_list)
stats_raw = np.concatenate(stat_seq_list)

sort_idx = np.lexsort((stats_raw, times_raw))
X_train_raw = X_train_raw[sort_idx]
y_train_raw = y_train_raw[sort_idx]
times_raw = times_raw[sort_idx]
stats_raw = stats_raw[sort_idx]


print(f"Done: Data Preprocessing complete! Generated {len(X_train_raw)} total sequences.")


 1. ADVANCED FEATURE ENGINEERING & REGIONAL SPATIAL METRICS
Starting: Train dataset preprocessing and GRU sequence generation...
  - Creating GRU sequences for each station...
    ... processed 4/12 stations
    ... processed 8/12 stations
    ... processed 12/12 stations
Done: Data Preprocessing complete! Generated 315360 total sequences.


In [3]:
# ==========================================
# 3. TABULAR FEATURES
# ==========================================
print("\nStarting: Tabular feature engineering for LightGBM...")

def create_lgb_tabular_features(df, seq_length=24):
    df_lgb = df.copy()
    rolling_cols = {}

    for window in [3, 6, 12, 24]:
        for col in ["PM2.5", "PM10", "TEMP", "DEWP", "U_wind", "V_wind"]:
            rolling_cols[f"{col}_mean_{window}"] = df_lgb.groupby(
                "station", observed=False
            )[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
            rolling_cols[f"{col}_std_{window}"] = df_lgb.groupby(
                "station", observed=False
            )[col].transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
            rolling_cols[f"{col}_max_{window}"] = df_lgb.groupby(
                "station", observed=False
            )[col].transform(lambda x: x.rolling(window, min_periods=1).max())
            rolling_cols[f"{col}_min_{window}"] = df_lgb.groupby(
                "station", observed=False
            )[col].transform(lambda x: x.rolling(window, min_periods=1).min())
            rolling_cols[f"{col}_ewm_{window}"] = df_lgb.groupby(
                "station", observed=False
            )[col].transform(lambda x: x.ewm(span=window, min_periods=1).mean())

    pm25_diff1 = df_lgb.groupby("station", observed=False)["PM2.5"].diff().fillna(0)
    rolling_cols["PM2.5_diff_1"] = pm25_diff1
    rolling_cols["PM2.5_diff_2"] = (
        pm25_diff1.groupby(df_lgb["station"], observed=False).diff().fillna(0)
    )
    rolling_cols["PM10_diff_1"] = (
        df_lgb.groupby("station", observed=False)["PM10"].diff().fillna(0)
    )
    rolling_cols["temp_dewp_spread"] = df_lgb["TEMP"] - df_lgb["DEWP"]
    rolling_cols["pm_ratio"] = (df_lgb["PM2.5"] / (df_lgb["PM10"] + 1e-5)).fillna(0)

    roll_feat_names = list(rolling_cols.keys()) + [
        "regional_pm25_mean",
        "station_regional_delta",
    ]
    df_lgb = pd.concat([df_lgb, pd.DataFrame(rolling_cols, index=df_lgb.index)], axis=1)

    xs_lgb = []
    lag_arr = df_lgb[LAG_FEATURES].values.astype(np.float32)
    roll_arr = df_lgb[roll_feat_names].values.astype(np.float32)
    target_time_arr = df_lgb[["month_sin", "month_cos", "hour_sin", "hour_cos"]].values
    stat_arr = df_lgb["station_code"].values

    for i in range(len(df_lgb) - seq_length):
        x_lags = lag_arr[i : i + seq_length].flatten()
        x_roll = roll_arr[i + seq_length - 1]
        x_time = target_time_arr[i + seq_length]
        x_stat = np.array([stat_arr[i + seq_length]])
        xs_lgb.append(np.concatenate([x_lags, x_roll, x_time, x_stat]))

    lgb_feature_names = []
    for lag in range(seq_length, 0, -1):
        for f in LAG_FEATURES:
            lgb_feature_names.append(f"{f}_lag_{lag}")
    lgb_feature_names.extend(roll_feat_names)
    lgb_feature_names.extend(
        ["month_sin", "month_cos", "hour_sin", "hour_cos", "station_code"]
    )

    return np.array(xs_lgb), lgb_feature_names


print("  - Creating LightGBM tabular features for each station...")
X_lgb_raw_list = []
for i, station in enumerate(train_df["station"].unique()):
    st_df = train_df[train_df["station"] == station]
    X_lgb_st, lgb_feature_names = create_lgb_tabular_features(st_df, SEQ_LENGTH)
    X_lgb_raw_list.append(X_lgb_st)
    if (i + 1) % 4 == 0:
        print(f"    ... processed {i + 1}/{len(train_df['station'].unique())} stations")

X_train_lgb_all = np.vstack(X_lgb_raw_list)[sort_idx]

print(f"Done: Tabular Features complete! Generated {len(X_train_lgb_all)} rows with {X_train_lgb_all.shape[1]} features.")


Starting: Tabular feature engineering for LightGBM...
  - Creating LightGBM tabular features for each station...
    ... processed 4/12 stations
    ... processed 8/12 stations
    ... processed 12/12 stations
Done: Tabular Features complete! Generated 315360 rows with 468 features.


In [4]:
# ==========================================
# PREPARE test_raw.csv FOR EVALUATION
# ==========================================
print("\n" + "=" * 50)
print(" PREPARING test_raw.csv FOR EVALUATION")
print("=" * 50)
print("Starting: test_raw.csv evaluation dataset preparation...")

test_raw_df = pd.read_csv(TEST_RAW_DATA_PATH)
test_raw_df["datetime"] = pd.to_datetime(test_raw_df[["year", "month", "day", "hour"]])
test_raw_df.sort_values(by=["station", "datetime"], inplace=True)
test_raw_df.reset_index(drop=True, inplace=True)

test_raw_df = engineer_features_narrow(test_raw_df)

test_regional_pm25 = test_raw_df.groupby("datetime")["PM2.5"].transform("mean")
test_raw_df["regional_pm25_mean"] = test_regional_pm25.ffill()
test_raw_df["station_regional_delta"] = (
    test_raw_df["PM2.5"] - test_raw_df["regional_pm25_mean"]
)

test_raw_df["station_code"] = test_raw_df["station"].map(station_map)

for feat in CONTINUOUS_FEATURES_LGB + ["regional_pm25_mean", "station_regional_delta"]:
    if feat in test_raw_df.columns:
        test_raw_df[feat] = test_raw_df.groupby("station", observed=False)[
            feat
        ].transform(lambda x: x.ffill())

for feat in MISSING_FEATURES_GRU:
    test_raw_df[feat] = test_raw_df[feat].fillna(0.0)

print("  - Creating GRU sequences for test_raw...")
(
    X_seq_test_raw_list,
    y_seq_test_raw_list,
    time_seq_test_raw_list,
    stat_seq_test_raw_list,
) = ([], [], [], [])
for station in test_raw_df["station"].unique():
    st_df = test_raw_df[test_raw_df["station"] == station]
    X_s, y_s, t_s, s_s = create_sequences_gru(st_df, SEQ_LENGTH)
    X_seq_test_raw_list.append(X_s)
    y_seq_test_raw_list.append(y_s)
    time_seq_test_raw_list.append(t_s)
    stat_seq_test_raw_list.append(s_s)

X_test_raw_gru = np.vstack(X_seq_test_raw_list)
y_true_test_raw = np.concatenate(y_seq_test_raw_list)
times_test_raw = np.concatenate(time_seq_test_raw_list)
stats_test_raw = np.concatenate(stat_seq_test_raw_list)

sort_idx_test = np.lexsort((stats_test_raw, times_test_raw))
X_test_raw_gru = X_test_raw_gru[sort_idx_test]
y_true_test_raw = y_true_test_raw[sort_idx_test]

print("  - Creating LightGBM features for test_raw...")
X_lgb_test_raw_list = []
for station in test_raw_df["station"].unique():
    st_df = test_raw_df[test_raw_df["station"] == station]
    X_lgb_st, _ = create_lgb_tabular_features(st_df, SEQ_LENGTH)
    X_lgb_test_raw_list.append(X_lgb_st)

X_test_raw_lgb = np.vstack(X_lgb_test_raw_list)[sort_idx_test]


print(f"Done: test_raw.csv evaluation prep complete! Generated {len(X_test_raw_gru)} sequences.")


 PREPARING test_raw.csv FOR EVALUATION
Starting: test_raw.csv evaluation dataset preparation...
  - Creating GRU sequences for test_raw...
  - Creating LightGBM features for test_raw...
Done: test_raw.csv evaluation prep complete! Generated 104832 sequences.


In [5]:
# ==========================================
# 4. PREPARE WIDE TEST DATASET
# ==========================================
print("\n" + "=" * 50)
print(" 2. PREPARING WIDE TEST DATASET (test.csv)")
print("=" * 50)
print("Starting: test.csv wide dataset preparation...")

test_df = pd.read_csv(TEST_DATA_PATH)

try:
    regional_test_pm25 = test_df.groupby(
        ["year_lag_1", "month_lag_1", "day_lag_1", "hour_lag_1"]
    )["PM2.5_lag_1"].transform("mean")
except KeyError:
    regional_test_pm25 = test_df["PM2.5_lag_1"]

test_df["regional_pm25_mean"] = regional_test_pm25.fillna(test_df["PM2.5_lag_1"])
test_df["station_regional_delta"] = (
    test_df["PM2.5_lag_1"] - test_df["regional_pm25_mean"]
)

print("  - Building lagged rolling windows for test.csv...")
for i in range(1, SEQ_LENGTH + 1):
    suffix = f"_lag_{i}"
    if f"wd{suffix}" in test_df.columns and f"WSPM{suffix}" in test_df.columns:
        angles_rad = np.radians(test_df[f"wd{suffix}"].map(WD_MAPPING).fillna(0))
        wspm = test_df[f"WSPM{suffix}"].fillna(0)
        test_df[f"U_wind{suffix}"] = -wspm * np.sin(angles_rad)
        test_df[f"V_wind{suffix}"] = -wspm * np.cos(angles_rad)
    if f"TEMP{suffix}" in test_df.columns and f"DEWP{suffix}" in test_df.columns:
        test_df[f"rel_humidity{suffix}"] = compute_relative_humidity(
            test_df[f"TEMP{suffix}"].fillna(0), test_df[f"DEWP{suffix}"].fillna(0)
        )
    if f"WSPM{suffix}" in test_df.columns and f"PM2.5{suffix}" in test_df.columns:
        test_df[f"dispersion_idx{suffix}"] = test_df[f"WSPM{suffix}"].fillna(
            0
        ) * test_df[f"PM2.5{suffix}"].fillna(0)
    if f"hour{suffix}" in test_df.columns:
        test_df[f"hour_sin{suffix}"] = np.sin(
            2 * np.pi * test_df[f"hour{suffix}"] / 24.0
        )
        test_df[f"hour_cos{suffix}"] = np.cos(
            2 * np.pi * test_df[f"hour{suffix}"] / 24.0
        )
    if f"month{suffix}" in test_df.columns:
        test_df[f"month_sin{suffix}"] = np.sin(
            2 * np.pi * (test_df[f"month{suffix}"] - 1) / 12.0
        )
        test_df[f"month_cos{suffix}"] = np.cos(
            2 * np.pi * (test_df[f"month{suffix}"] - 1) / 12.0
        )
    if f"PM2.5{suffix}" in test_df.columns:
        try:
            reg_mean = test_df.groupby(
                [f"year{suffix}", f"month{suffix}", f"day{suffix}", f"hour{suffix}"]
            )[f"PM2.5{suffix}"].transform("mean")
        except KeyError:
            reg_mean = test_df[f"PM2.5{suffix}"]
        test_df[f"regional_pm25_mean{suffix}"] = reg_mean.fillna(
            test_df[f"PM2.5{suffix}"]
        )
        test_df[f"station_regional_delta{suffix}"] = (
            test_df[f"PM2.5{suffix}"] - test_df[f"regional_pm25_mean{suffix}"]
        )

test_df["station"] = test_df["station"].astype("category")
test_df["station_code"] = test_df["station"].map(station_map)

try:
    target_time = pd.to_datetime(
        test_df[["year_lag_1", "month_lag_1", "day_lag_1", "hour_lag_1"]].rename(
            columns={
                "year_lag_1": "year",
                "month_lag_1": "month",
                "day_lag_1": "day",
                "hour_lag_1": "hour",
            }
        )
    ) + pd.Timedelta(hours=1)
    t_month, t_hour = target_time.dt.month, target_time.dt.hour
except KeyError:
    t_month = test_df["month_lag_1"]
    t_hour = (test_df["hour_lag_1"] + 1) % 24

test_engineered = {}
test_engineered["month_sin"] = np.sin(2 * np.pi * (t_month - 1) / 12.0)
test_engineered["month_cos"] = np.cos(2 * np.pi * (t_month - 1) / 12.0)
test_engineered["hour_sin"] = np.sin(2 * np.pi * t_hour / 24.0)
test_engineered["hour_cos"] = np.cos(2 * np.pi * t_hour / 24.0)

print("  - Calculating moving averages for test.csv...")
for w in [3, 6, 12, 24]:
    for var_name in ["PM2.5", "PM10", "TEMP", "DEWP", "U_wind", "V_wind"]:
        var_cols = [
            f"{var_name}_lag_{i}"
            for i in range(1, w + 1)
            if f"{var_name}_lag_{i}" in test_df.columns
        ]
        if var_cols:
            sub_df = test_df[var_cols]
            test_engineered[f"{var_name}_mean_{w}"] = sub_df.mean(axis=1)
            test_engineered[f"{var_name}_std_{w}"] = sub_df.std(axis=1).fillna(0)
            test_engineered[f"{var_name}_max_{w}"] = sub_df.max(axis=1)
            test_engineered[f"{var_name}_min_{w}"] = sub_df.min(axis=1)
            alpha = 2.0 / (w + 1.0)
            weights = (1.0 - alpha) ** np.arange(len(var_cols))
            weights /= weights.sum()
            test_engineered[f"{var_name}_ewm_{w}"] = (sub_df.values * weights).sum(
                axis=1
            )

test_engineered["PM2.5_diff_1"] = test_df["PM2.5_lag_1"] - test_df["PM2.5_lag_2"]
test_engineered["PM2.5_diff_2"] = (test_df["PM2.5_lag_1"] - test_df["PM2.5_lag_2"]) - (
    test_df["PM2.5_lag_2"] - test_df["PM2.5_lag_3"]
)
test_engineered["PM10_diff_1"] = test_df["PM10_lag_1"] - test_df["PM10_lag_2"]
test_engineered["temp_dewp_spread"] = test_df["TEMP_lag_1"] - test_df["DEWP_lag_1"]
test_engineered["pm_ratio"] = (
    test_df["PM2.5_lag_1"] / (test_df["PM10_lag_1"] + 1e-5)
).fillna(0)

test_df = pd.concat(
    [test_df, pd.DataFrame(test_engineered, index=test_df.index)], axis=1
)

test_missing_cols = {}
for i in range(1, SEQ_LENGTH + 1):
    for feat in MISSING_TARGETS:
        lag_col = f"{feat}_lag_{i}"
        if lag_col in test_df.columns:
            test_missing_cols[f"{feat}_is_missing_lag_{i}"] = (
                test_df[lag_col].isna().astype(float)
            )

if test_missing_cols:
    test_df = pd.concat(
        [test_df, pd.DataFrame(test_missing_cols, index=test_df.index)], axis=1
    )

# Sequence Test filling logic
for f in CONTINUOUS_FEATURES_GRU:
    cols = [f"{f}_lag_{i}" for i in range(SEQ_LENGTH, 0, -1)]
    avail = [c for c in cols if c in test_df.columns]
    if avail:
        test_df[avail] = test_df[avail].ffill(axis=1).bfill(axis=1)

for f in MISSING_FEATURES_GRU:
    cols = [f"{f}_lag_{i}" for i in range(SEQ_LENGTH, 0, -1)]
    avail = [c for c in cols if c in test_df.columns]
    if avail:
        test_df[avail] = test_df[avail].fillna(0.0)

X_test_gru = np.zeros((len(test_df), SEQ_LENGTH, len(FEATURES_GRU)))
for f_idx, feature in enumerate(FEATURES_GRU):
    cols = [f"{feature}_lag_{i}" for i in range(SEQ_LENGTH, 0, -1)]
    X_test_gru[:, :, f_idx] = test_df[cols].values

# Tabular Test filling logic
for f in CONTINUOUS_FEATURES_LGB:
    cols = [f"{f}_lag_{i}" for i in range(SEQ_LENGTH, 0, -1)]
    avail = [c for c in cols if c in test_df.columns]
    if avail:
        test_df[avail] = test_df[avail].ffill(axis=1).bfill(axis=1)

X_test_lgb = test_df[lgb_feature_names].values


print("Done: Wide Test Dataset prep complete!")


 2. PREPARING WIDE TEST DATASET (test.csv)
Starting: test.csv wide dataset preparation...
  - Building lagged rolling windows for test.csv...
  - Calculating moving averages for test.csv...
Done: Wide Test Dataset prep complete!


In [6]:
# ==========================================
# 5. GRU MODEL DEFINITION
# ==========================================
print("Starting: Defining GRU architecture...")

class ResidualAirQualityGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(ResidualAirQualityGRU, self).__init__()
        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        out, _ = self.gru(x)
        out_last = self.layer_norm(out[:, -1, :])
        return self.head(out_last)


print("Done: GRU Model defined!")

Starting: Defining GRU architecture...
Done: GRU Model defined!


In [7]:
# ==========================================
# 6. ENSEMBLE TRAINING & BLENDING
# ==========================================
print("\n" + "=" * 50)
print(f" 3. EXECUTING {N_FOLDS}-FOLD CV FOR ENSEMBLE")
print("=" * 50)
print("Starting: 5-Fold CV Ensemble Training...")

unique_times = np.sort(np.unique(times_raw))
total_timesteps = len(unique_times)
block_size = total_timesteps // N_FOLDS

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.02,
    "num_leaves": 127,
    "max_depth": 8,
    "min_child_samples": 30,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 1.5,
    "lambda_l2": 3.0,
    "force_col_wise": True,
    "num_threads": 2,
    "verbose": -1,
    "random_state": SEED,
}

oof_preds_lgb = np.zeros(len(y_train_raw))
oof_preds_gru = np.zeros(len(y_train_raw))

test_preds_lgb_folds = np.zeros((N_FOLDS, len(test_df)))
test_preds_gru_folds = np.zeros((N_FOLDS, len(test_df)))
test_raw_preds_lgb_folds = np.zeros((N_FOLDS, len(X_test_raw_lgb)))
test_raw_preds_gru_folds = np.zeros((N_FOLDS, len(X_test_raw_gru)))

for fold in range(N_FOLDS):
    print(f"\n--- RUNNING FOLD {fold+1}/{N_FOLDS} ---")
    val_time_start = unique_times[fold * block_size]
    val_time_end = (
        unique_times[(fold + 1) * block_size - 1]
        if fold < N_FOLDS - 1
        else unique_times[-1]
    )

    val_mask = (times_raw >= val_time_start) & (times_raw <= val_time_end)
    purge_start = val_time_start - np.timedelta64(PURGE_GAP_HOURS, "h")
    purge_end = val_time_end + np.timedelta64(PURGE_GAP_HOURS, "h")
    train_mask = (times_raw < purge_start) | (times_raw > purge_end)

    val_indices = np.where(val_mask)[0]
    train_indices = np.where(train_mask)[0]

    # -----------------------
    # Train Deep GRU
    # -----------------------
    print("  -> Training Deep GRU...")
    tr_feats = X_train_raw[train_indices].reshape(-1, len(FEATURES_GRU))
    feat_mean = tr_feats.mean(axis=0, dtype=np.float32)
    feat_std = tr_feats.std(axis=0, dtype=np.float32)
    feat_std[feat_std == 0] = 1.0
    del tr_feats

    X_tr_scaled = X_train_raw[train_indices].copy()
    X_tr_scaled -= feat_mean
    X_tr_scaled /= feat_std

    X_val_scaled = X_train_raw[val_indices].copy()
    X_val_scaled -= feat_mean
    X_val_scaled /= feat_std

    X_test_scaled = X_test_gru.copy()
    X_test_scaled -= feat_mean
    X_test_scaled /= feat_std
    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0).astype(np.float32)

    X_test_raw_scaled = X_test_raw_gru.copy()
    X_test_raw_scaled -= feat_mean
    X_test_raw_scaled /= feat_std
    X_test_raw_scaled = np.nan_to_num(X_test_raw_scaled, nan=0.0).astype(np.float32)

    target_idx = FEATURES_GRU.index(TARGET)
    target_mean = feat_mean[target_idx]
    target_std = feat_std[target_idx]

    y_tr_scaled = (y_train_raw[train_indices] - target_mean) / target_std
    y_val_scaled = (y_train_raw[val_indices] - target_mean) / target_std

    X_tr_t = torch.from_numpy(X_tr_scaled)
    y_tr_t = torch.from_numpy(y_tr_scaled).unsqueeze(1)
    train_loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t), batch_size=BATCH_SIZE, shuffle=True
    )

    X_v_t = torch.from_numpy(X_val_scaled)
    y_v_t = torch.from_numpy(y_val_scaled).unsqueeze(1)
    val_loader = DataLoader(
        TensorDataset(X_v_t, y_v_t), batch_size=BATCH_SIZE, shuffle=False
    )

    X_te_t = torch.from_numpy(X_test_scaled)
    test_loader = DataLoader(
        TensorDataset(X_te_t), batch_size=BATCH_SIZE, shuffle=False
    )

    X_te_raw_t = torch.from_numpy(X_test_raw_scaled)
    test_raw_loader = DataLoader(
        TensorDataset(X_te_raw_t), batch_size=BATCH_SIZE, shuffle=False
    )

    gru_model = ResidualAirQualityGRU(
        len(FEATURES_GRU), HIDDEN_SIZE_GRU, NUM_LAYERS_GRU
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(gru_model.parameters(), lr=LR_GRU, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS_GRU, eta_min=1e-5
    )
    criterion = nn.SmoothL1Loss(beta=1.0)

    use_amp = DEVICE.type == "cuda"
    amp_scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_loss = float("inf")
    patience_cnt = 0
    best_weights_path = f"gru_fold_{fold+1}.pt"

    for epoch in range(EPOCHS_GRU):
        gru_model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    preds = gru_model(X_b)
                    loss = criterion(preds, y_b)
                amp_scaler.scale(loss).backward()
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                preds = gru_model(X_b)
                loss = criterion(preds, y_b)
                loss.backward()
                optimizer.step()

        scheduler.step()
        gru_model.eval()
        v_preds, v_targets = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b = X_b.to(DEVICE)
                if use_amp:
                    with torch.cuda.amp.autocast():
                        preds = gru_model(X_b)
                else:
                    preds = gru_model(X_b)
                v_preds.extend(preds.cpu().numpy().flatten())
                v_targets.extend(y_b.cpu().numpy().flatten())

        val_rmse_scaled = np.sqrt(mean_squared_error(v_targets, v_preds))

        if val_rmse_scaled < best_val_loss:
            best_val_loss = val_rmse_scaled
            patience_cnt = 0
            torch.save(gru_model.state_dict(), best_weights_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE_GRU:
                break

    gru_model.load_state_dict(torch.load(best_weights_path, weights_only=True))
    gru_model.eval()

    v_preds_unscaled = []
    with torch.no_grad():
        for X_b, _ in val_loader:
            X_b = X_b.to(DEVICE)
            if use_amp:
                with torch.cuda.amp.autocast():
                    preds = gru_model(X_b)
            else:
                preds = gru_model(X_b)
            v_preds_unscaled.extend(
                ((preds * target_std) + target_mean).cpu().numpy().flatten()
            )
    oof_preds_gru[val_indices] = np.clip(np.array(v_preds_unscaled), 0, None)

    t_preds_unscaled = []
    t_raw_preds_unscaled = []
    with torch.no_grad():
        for (X_b,) in test_loader:
            X_b = X_b.to(DEVICE)
            if use_amp:
                with torch.cuda.amp.autocast():
                    preds = gru_model(X_b)
            else:
                preds = gru_model(X_b)
            t_preds_unscaled.extend(
                ((preds * target_std) + target_mean).cpu().numpy().flatten()
            )
        for (X_b,) in test_raw_loader:
            X_b = X_b.to(DEVICE)
            if use_amp:
                with torch.cuda.amp.autocast():
                    preds = gru_model(X_b)
            else:
                preds = gru_model(X_b)
            t_raw_preds_unscaled.extend(
                ((preds * target_std) + target_mean).cpu().numpy().flatten()
            )
    test_preds_gru_folds[fold] = np.array(t_preds_unscaled)
    test_raw_preds_gru_folds[fold] = np.array(t_raw_preds_unscaled)

    gru_fold_rmse = np.sqrt(
        mean_squared_error(y_train_raw[val_indices], oof_preds_gru[val_indices])
    )
    print(f"     [GRU]  Fold {fold+1} Val RMSE: {gru_fold_rmse:.4f}")

    # Free memory
    del X_tr_scaled, X_val_scaled, X_test_scaled, X_test_raw_scaled
    del X_tr_t, y_tr_t, X_v_t, y_v_t, X_te_t, X_te_raw_t
    del train_loader, val_loader, test_loader, test_raw_loader
    del gru_model, optimizer, scheduler, criterion
    import gc

    gc.collect()

    # -----------------------
    # Train LightGBM
    # -----------------------
    print("  -> Training LightGBM...")
    stat_idx = lgb_feature_names.index("station_code")
    lgb_tr_data = lgb.Dataset(
        X_train_lgb_all[train_indices],
        label=y_train_raw[train_indices],
        feature_name=lgb_feature_names,
        categorical_feature=[stat_idx],
    )
    lgb_val_data = lgb.Dataset(
        X_train_lgb_all[val_indices],
        label=y_train_raw[val_indices],
        feature_name=lgb_feature_names,
        reference=lgb_tr_data,
        categorical_feature=[stat_idx],
    )

    lgb_model = lgb.train(
        lgb_params,
        lgb_tr_data,
        num_boost_round=1000,
        valid_sets=[lgb_tr_data, lgb_val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    oof_preds_lgb[val_indices] = np.clip(
        lgb_model.predict(X_train_lgb_all[val_indices]), 0, None
    )
    test_preds_lgb_folds[fold] = lgb_model.predict(X_test_lgb)
    test_raw_preds_lgb_folds[fold] = lgb_model.predict(X_test_raw_lgb)

    lgb_fold_rmse = np.sqrt(
        mean_squared_error(y_train_raw[val_indices], oof_preds_lgb[val_indices])
    )
    print(f"     [LGBM] Fold {fold+1} Val RMSE: {lgb_fold_rmse:.4f}")

    # Free LightGBM memory
    del lgb_tr_data, lgb_val_data, lgb_model
    gc.collect()

print("\n" + "=" * 50)
print(" 4. ENSEMBLE EVALUATION & BLENDING OPTIMIZATION")
print("=" * 50)

avg_test_preds_lgb = np.clip(test_preds_lgb_folds.mean(axis=0), 0, None)
avg_test_preds_gru = np.clip(test_preds_gru_folds.mean(axis=0), 0, None)

avg_test_raw_preds_lgb = np.clip(test_raw_preds_lgb_folds.mean(axis=0), 0, None)
avg_test_raw_preds_gru = np.clip(test_raw_preds_gru_folds.mean(axis=0), 0, None)

oof_lgb_rmse = np.sqrt(mean_squared_error(y_train_raw, oof_preds_lgb))
test_lgb_rmse = np.sqrt(mean_squared_error(y_true_test_raw, avg_test_raw_preds_lgb))

oof_gru_rmse = np.sqrt(mean_squared_error(y_train_raw, oof_preds_gru))
test_gru_rmse = np.sqrt(mean_squared_error(y_true_test_raw, avg_test_raw_preds_gru))

print(
    f"Standalone LightGBM : OOF RMSE: {oof_lgb_rmse:.4f} | Test RMSE: {test_lgb_rmse:.4f}"
)
print(
    f"Standalone Deep GRU : OOF RMSE: {oof_gru_rmse:.4f} | Test RMSE: {test_gru_rmse:.4f}"
)

# Grid Search for best blend weight
print("\nSearching for optimal ensemble weights based on OOF performance...")
best_w = 0.0
best_rmse = float("inf")

for w_int in range(101):
    w = w_int / 100.0
    ensemble_oof = np.clip(w * oof_preds_lgb + (1 - w) * oof_preds_gru, 0, None)
    rmse = np.sqrt(mean_squared_error(y_train_raw, ensemble_oof))
    if rmse < best_rmse:
        best_rmse = rmse
        best_w = w

print(f"Optimal Weight Identified: {best_w:.2f} * LightGBM + {1 - best_w:.2f} * GRU")

# Evaluate combined test
ensemble_test_preds = best_w * avg_test_preds_lgb + (1 - best_w) * avg_test_preds_gru

ensemble_test_raw_preds = (
    best_w * avg_test_raw_preds_lgb + (1 - best_w) * avg_test_raw_preds_gru
)
ensemble_test_rmse = np.sqrt(
    mean_squared_error(y_true_test_raw, ensemble_test_raw_preds)
)
ensemble_test_mae = mean_absolute_error(y_true_test_raw, ensemble_test_raw_preds)

print(
    f"\nFinal Ensemble | OOF RMSE: {best_rmse:.4f} | Test RMSE: {ensemble_test_rmse:.4f} | Test MAE: {ensemble_test_mae:.4f}"
)

submission_df = pd.DataFrame({"id": test_df["id"], "PM2.5": ensemble_test_preds})
submission_df.to_csv("submission.csv", index=False)
print("Saved predictions to 'submission.csv'!")

print("Done: Ensemble Training and Submission complete!")


 3. EXECUTING 5-FOLD CV FOR ENSEMBLE
Starting: 5-Fold CV Ensemble Training...

--- RUNNING FOLD 1/5 ---
  -> Training Deep GRU...
     [GRU]  Fold 1 Val RMSE: 18.6678
  -> Training LightGBM...
     [LGBM] Fold 1 Val RMSE: 18.5054

--- RUNNING FOLD 2/5 ---
  -> Training Deep GRU...
     [GRU]  Fold 2 Val RMSE: 19.8603
  -> Training LightGBM...
     [LGBM] Fold 2 Val RMSE: 19.8587

--- RUNNING FOLD 3/5 ---
  -> Training Deep GRU...
     [GRU]  Fold 3 Val RMSE: 13.9149
  -> Training LightGBM...
     [LGBM] Fold 3 Val RMSE: 13.8093

--- RUNNING FOLD 4/5 ---
  -> Training Deep GRU...
     [GRU]  Fold 4 Val RMSE: 17.3112
  -> Training LightGBM...
     [LGBM] Fold 4 Val RMSE: 16.8999

--- RUNNING FOLD 5/5 ---
  -> Training Deep GRU...
     [GRU]  Fold 5 Val RMSE: 19.2422
  -> Training LightGBM...
     [LGBM] Fold 5 Val RMSE: 20.0272

 4. ENSEMBLE EVALUATION & BLENDING OPTIMIZATION
Standalone LightGBM : OOF RMSE: 17.9678 | Test RMSE: 17.1098
Standalone Deep GRU : OOF RMSE: 17.9248 | Test RMSE